In [3]:
from langchain.docstore.document import Document
import os
import re
import streamlit as st
from itertools import permutations

db_id_dict = {}
def init_document():
    db_id_dict = {}
    documents = []
    directory_path = 'C:\Research-Paper\PAPER-WORK-2024\RAG-FILES'
    # The directory contains documents in format <TABLE_NAME>__<DB_ID>.txt
    files = os.listdir(directory_path)

    for file in files:
        file_path = os.path.join(directory_path, file)
        if os.path.isfile(file_path):
            with open(file_path, 'r') as f:
                print("Reading file => ",file_path)
                match = re.search(r"(.*?)__(.*?)\.txt", file)
                table_name = match.group(1)
                db_id = match.group(2)
                if db_id not in db_id_dict:
                    db_id_dict[db_id] = [table_name]
                else:
                    db_id_dict[db_id].append(table_name)
                documents.append(Document(page_content=f.read(), metadata={"source": "local", "context": db_id,"table name":table_name}))
    print("Number of documents currently used: ", len(documents))
    return documents
    
    
documents = init_document()

Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Attribute_Definitions__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\bank__loan_1.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalogs__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalog_Contents_Additional_Attributes__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalog_Contents__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalog_Structure__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\church__wedding.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\city__farm.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Claims__insurance_policies.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\climber__climbing.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\competition_re

In [22]:
print(db_id_dict)

{}


In [23]:
len(documents)

54

# Embeddings from:

https://huggingface.co/spaces/mteb/leaderboard

TAG PAPER: https://arxiv.org/abs/2212.03533

In [4]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings # Source: https://medium.com/international-school-of-ai-data-science/implementing-rag-with-langchain-and-hugging-face-28e3ea66c5f7

def init_embedding():
    print("Init embeddings...")
    # Define the path to the pre-trained model you want to use
    # modelPath = "sentence-transformers/all-MiniLM-l6-v2"
    # modelPath = "sentence-transformers/all-MPNet-base-v2"
    modelPath = "sentence-transformers/all-MiniLM-L12-v2" # This is a lightweight and fast model with good performance on semantic textual similarity tasks
    #modelPath = "dunzhang/stella_en_1.5B_v5" #https://huggingface.co/spaces/mteb/leaderboard # THIS IS WORKING

    
    #modelPath = "nvidia/NV-Embed-v2"
    modelPath = "dunzhang/stella_en_1.5B_v5" #https://huggingface.co/spaces/mteb/leaderboard # THIS IS WORKING

    modelPath = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
    modelPath = "intfloat/e5-mistral-7b-instruct"
    #modelPath = "intfloat/e5-base"

    modelPath = "intfloat/e5-base-v2"

    # TESTS
    modelPath = "sentence-transformers/all-MiniLM-L12-v2"
    modelPath = "Alibaba-NLP/gte-large-en-v1.5"
    

    
    modelPath = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
    modelPath = "sentence-transformers/all-MiniLM-L12-v2"

    modelPath = "intfloat/e5-base-v2"
    modelPath = "dunzhang/stella_en_1.5B_v5"

    modelPath = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"

    #modelPath = "intfloat/e5-mistral-7b-instruct"
    modelPath = "sentence-transformers/all-MiniLM-L12-v2"

    
    # Create a dictionary with model configuration options, specifying to use the CPU for computations
    model_kwargs = {'device': 'cpu'}
    model_kwargs = {'device': 'cpu', 'trust_remote_code': True} 
    model_kwargs = {'device': 'cpu'}

    # Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
    encode_kwargs = {'normalize_embeddings': False}
    

    print("LOADING : ",modelPath)
    # Initialize an instance of HuggingFaceEmbeddings with the specified parameters
    """embeddings = HuggingFaceEmbeddings(
        model_name=modelPath,  # Provide the pre-trained model's path
        model_kwargs=model_kwargs,  # Pass the model configuration options
        encode_kwargs=encode_kwargs  # Pass the encoding options
    )"""
    embeddings = HuggingFaceEmbeddings(
        model_name=modelPath,  # Provide the pre-trained model's path
        model_kwargs=model_kwargs,  # Pass the model configuration options
        encode_kwargs=encode_kwargs,  # Pass the encoding options
    )
    print("LOADED : ",modelPath)
    return embeddings

def init_embedding():
    print("Init embeddings...")
    # Define the path to the pre-trained model you want to use
    # This is a lightweight and fast model with good performance on semantic textual similarity tasks
    model_path = "sentence-transformers/all-MiniLM-L12-v2"
    #model_path = "dunzhang/stella_en_1.5B_v5"

    """
        Currently the above embedding works fine, but in future we can use more embeddings from https://huggingface.co/spaces/mteb/leaderboard
        As the data grows, some of the embeddings we have tested already are below we can test again these on new data and try among these:
            1. sentence-transformers/all-MiniLM-L12-v2 (RANK: 123)
            2. Alibaba-NLP/gte-large-en-v1.5 (RANK: 19)
            3. dunzhang/stella_en_1.5B_v5 (RANK: 3)
            4. Alibaba-NLP/gte-Qwen2-1.5B-instruct (RANK: 13)
            5. intfloat/e5-base-v2 (USED IN TAG PAPER: https://arxiv.org/pdf/2408.14717)
    """

    # Create a dictionary with model configuration options, specifying to use the CPU for computations
    model_kwargs = {'device': 'cpu'}
    #model_kwargs = {'device': 'cpu', 'trust_remote_code': True} # Sometime this config works for other embeddings in list above

    # Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
    encode_kwargs = {'normalize_embeddings': False}

    # Initialize an instance of HuggingFaceEmbeddings with the specified parameters
    embeddings = HuggingFaceEmbeddings(
        model_name=model_path,  # Provide the pre-trained model's path
        model_kwargs=model_kwargs,  # Pass the model configuration options
        encode_kwargs=encode_kwargs  # Pass the encoding options
    )
    print("Loaded embedding : ",model_path)
    return embeddings

In [5]:
#init_embedding()

In [4]:
!pip show langchain

Name: langchain
Version: 0.1.12
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages
Requires: aiohttp, dataclasses-json, jsonpatch, langchain-community, langchain-core, langchain-text-splitters, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 


In [15]:
!pip install langchain==0.1.12

  Obtaining dependency information for langchain==0.1.12 from https://files.pythonhosted.org/packages/b0/58/9a8777dff52bf5485c391cf3951e3d3b787afdc5967b29e25694ea014377/langchain-0.1.12-py3-none-any.whl.metadata
  Using cached langchain-0.1.12-py3-none-any.whl.metadata (13 kB)
  Obtaining dependency information for langchain-community<0.1,>=0.0.28 from https://files.pythonhosted.org/packages/1b/d3/1f4d1941ae5a627299c8ea052847b99ad6674b97b699d8a08fc4faf25d3e/langchain_community-0.0.38-py3-none-any.whl.metadata
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Obtaining dependency information for langchain-core<0.2.0,>=0.1.31 from https://files.pythonhosted.org/packages/6a/10/285fa149ce95300d91ea0bb124eec28889e5ebbcb59434d1fe2f31098d72/langchain_core-0.1.53-py3-none-any.whl.metadata
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
  Obtaining dependency information for langchain-text-splitters<0.1,>=0.0.1 from https://files.pythonhosted

In [6]:
!pip install sentence-transformers

  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/8b/c8/990e22a465e4771338da434d799578865d6d7ef1fdb50bd844b7ecdcfa19/sentence_transformers-3.3.1-py3-none-any.whl.metadata
  Obtaining dependency information for transformers<5.0.0,>=4.41.0 from https://files.pythonhosted.org/packages/51/51/b87caa939fedf307496e4dbf412f4b909af3d9ca8b189fc3b65c1faa456f/transformers-4.46.3-py3-none-any.whl.metadata
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Obtaining dependency information for tokenizers<0.21,>=0.20 from https://files.pythonhosted.org/packages/75/68/1b4f928b15a36ed278332ac75d66d7eb65d865bf344d049c452c18447bf9/tokenizers-0.20.3-cp311-none-win_amd64.whl.metadata
  Using cached tokenizers-0.20.3-cp311-none-win_amd64.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/268.8 kB ? eta -:--:--
   ---------------------------------------  266.2/268.8 kB 8.3 MB/s eta 0:00:01
   -----------------------

In [5]:
# Global variable to store the cached result
cached_db = None
cached_embed = None


def init_database():
    print("Init database...")
    global cached_db
    print("CACHED_DB = ",cached_db)
    global cached_embed
    print("EMBED = ",cached_embed)

    if cached_embed is None:
        print("\n\nLOADING EMBEDDING.....\n\n")
        cached_embed = init_embedding()

    if True:
        # If the result is already cached, return it
        if cached_db is not None:
            print("Using cached database...")
            return cached_db
        documents = init_document()
        db = FAISS.from_documents(documents, cached_embed)
        # Cache the result
        cached_db = db
        return db

In [6]:
def similarity_k_search(question,k=3):
    db = init_database()
    #db,db_id_dict = init_database()
    searchDocs = db.similarity_search_with_score(question,k=k)
    print(len(searchDocs))
    # get a list with page content and score
    for i in range(k):
        print(searchDocs[i][0].metadata["table name"]," from db_id : ",searchDocs[i][0].metadata["context"]," scored ::>",searchDocs[i][1])
    return [(searchDocs[i][0].page_content,searchDocs[i][1],searchDocs[i][0].metadata["table name"],searchDocs[i][0].metadata["context"]) for i in range(k)]

In [7]:
question = "How many farms are there?"
answer = similarity_k_search(question,k=3)

Init database...
CACHED_DB =  None
EMBED =  None


LOADING EMBEDDING.....


Init embeddings...


C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
C:\Users\GuPr564\AppData\Local\anaconda3\Lib\site-packages\t

Loaded embedding :  sentence-transformers/all-MiniLM-L12-v2
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Attribute_Definitions__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\bank__loan_1.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalogs__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalog_Contents_Additional_Attributes__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalog_Contents__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Catalog_Structure__product_catalog.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\church__wedding.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\city__farm.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\Claims__insurance_policies.txt
Reading file =>  C:\Research-Paper\PAPER-WORK-2024\RAG-FILES\climber__climbing.txt
Reading file =>

In [8]:
answer

[('CREATE TABLE "farm" (\n"Farm_ID" int,\n"Year" int,\n"Total_Horses" real,\n"Working_Horses" real,\n"Total_Cattle" real,\n"Oxen" real,\n"Bulls" real,\n"Cows" real,\n"Pigs" real,\n"Sheep_and_Goats" real,\nPRIMARY KEY ("Farm_ID")\n);\n\n\nINSERT INTO  "farm" VALUES (1,"1927","5056.5","3900.1","8374.5","805.5","31.6","3852.1","4412.4","7956.3");\nINSERT INTO  "farm" VALUES (2,"1928","5486.9","4090.5","8604.8","895.3","32.8","3987.0","6962.9","8112.2");\nINSERT INTO  "farm" VALUES (3,"1929","5607.5","4198.8","7611.0","593.7","26.9","3873.0","4161.2","7030.8");\nINSERT INTO  "farm" VALUES (4,"1930","5308.2","3721.6","6274.1","254.8","49.6","3471.6","3171.8","4533.4");\nINSERT INTO  "farm" VALUES (5,"1931","4781.3","3593.7","6189.5","113.8","40.0","3377.0","3373.3","3364.8");\nINSERT INTO  "farm" VALUES (6,"1932","3658.9","3711.6","5006.7","105.2","71.6","2739.5","2623.7","2109.5");\nINSERT INTO  "farm" VALUES (7,"1933","2604.8","3711.2","4446.3","116.9","37.6","2407.2","2089.2","2004.7");\

In [46]:
question = 'What are the census rankings of cities that do not have the status "Village"?'
answer = similarity_k_search(question,k=3)

Init database...
CACHED_DB =  <langchain_community.vectorstores.faiss.FAISS object at 0x0000020D33C479D0>
EMBED =  client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='sentence-transformers/all-MiniLM-L12-v2' cache_folder=None model_kwargs={'device': 'cpu'} encode_kwargs={'normalize_embeddings': False} multi_process=False show_progress=False
Using cached database...
3
city  from db_id :  farm  scored ::> 1.1103858
county  from db_id :  election  scored ::> 1.3208994
bank  from db_id :  loan_1  scored ::> 1.4342859


In [47]:
answer

[('CREATE TABLE "city" (\n"City_ID" int,\n"Official_Name" text,\n"Status" text,\n"Area_km_2" real,\n"Population" real,\n"Census_Ranking" text,\nPRIMARY KEY ("City_ID")\n);\n\n\nINSERT INTO  "city" VALUES (1,"Grand Falls/Grand-Sault","Town","18.06","5706","636 of 5008");\nINSERT INTO  "city" VALUES (2,"Perth-Andover","Village","8.89","1778","1442 of 5,008");\nINSERT INTO  "city" VALUES (3,"Plaster Rock","Village","3.09","1135","1936 of 5,008");\nINSERT INTO  "city" VALUES (4,"Drummond","Village","8.91","775","2418 of 5008");\nINSERT INTO  "city" VALUES (5,"Aroostook","Village","2.24","351","3460 of 5008");\n',
  1.1103858,
  'city',
  'farm'),
 ('CREATE TABLE "county" (\n"County_Id" int,\n"County_name" text,\n"Population" real,\n"Zip_code" text,\nPRIMARY KEY ("County_Id")\n);\n\n\nINSERT INTO  "county" VALUES (1,"Howard",21000, "D21");\nINSERT INTO  "county" VALUES (2,"Baltimore County", 90000,"D08");\nINSERT INTO  "county" VALUES (3,"Colony",79000,"D02");\nINSERT INTO  "county" VALUES 

In [34]:
question = 'What are the official names of cities that have population over 1500 or less than 500?'
answer = similarity_k_search(question,k=3)

Init database...
CACHED_DB =  <langchain_community.vectorstores.faiss.FAISS object at 0x0000020D3BD50CD0>
EMBED =  client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='sentence-transformers/all-MiniLM-L12-v2' cache_folder=None model_kwargs={'device': 'cpu'} encode_kwargs={'normalize_embeddings': False} multi_process=False show_progress=False
Using cached database...
3
city  from db_id :  farm  scored ::> 1.2236102
county  from db_id :  election  scored ::> 1.3352318
market  from db_id :  film_rank  scored ::> 1.4378321


In [35]:
answer

[('CREATE TABLE "city" (\n"City_ID" int,\n"Official_Name" text,\n"Status" text,\n"Area_km_2" real,\n"Population" real,\n"Census_Ranking" text,\nPRIMARY KEY ("City_ID")\n);\n\n\nINSERT INTO  "city" VALUES (1,"Grand Falls/Grand-Sault","Town","18.06","5706","636 of 5008");\nINSERT INTO  "city" VALUES (2,"Perth-Andover","Village","8.89","1778","1442 of 5,008");\nINSERT INTO  "city" VALUES (3,"Plaster Rock","Village","3.09","1135","1936 of 5,008");\nINSERT INTO  "city" VALUES (4,"Drummond","Village","8.91","775","2418 of 5008");\nINSERT INTO  "city" VALUES (5,"Aroostook","Village","2.24","351","3460 of 5008");\n',
  1.2236102,
  'city',
  'farm'),
 ('CREATE TABLE "county" (\n"County_Id" int,\n"County_name" text,\n"Population" real,\n"Zip_code" text,\nPRIMARY KEY ("County_Id")\n);\n\n\nINSERT INTO  "county" VALUES (1,"Howard",21000, "D21");\nINSERT INTO  "county" VALUES (2,"Baltimore County", 90000,"D08");\nINSERT INTO  "county" VALUES (3,"Colony",79000,"D02");\nINSERT INTO  "county" VALUES 

In [36]:
question = 'What are the themes of competitions that have corresponding host cities with more than 1000 residents?'
answer = similarity_k_search(question,k=3)

Init database...
CACHED_DB =  <langchain_community.vectorstores.faiss.FAISS object at 0x0000020D3BD50CD0>
EMBED =  client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='sentence-transformers/all-MiniLM-L12-v2' cache_folder=None model_kwargs={'device': 'cpu'} encode_kwargs={'normalize_embeddings': False} multi_process=False show_progress=False
Using cached database...
3
farm_competition  from db_id :  farm  scored ::> 1.2192101
stadium  from db_id :  swimming  scored ::> 1.2509142
city  from db_id :  farm  scored ::> 1.2717694


In [39]:
answer

[('CREATE TABLE "farm_competition" (\n"Competition_ID" int,\n"Year" int,\n"Theme" text,\n"Host_city_ID" int,\n"Hosts" text,\nPRIMARY KEY ("Competition_ID"),\nFOREIGN KEY (`Host_city_ID`) REFERENCES `city`(`City_ID`)\n);\n\n\nINSERT INTO  "farm_competition" VALUES (1,"2013","Carnival M is back!",1,"Miley Cyrus Jared Leto and Karen Mok");\nINSERT INTO  "farm_competition" VALUES (2,"2006","Codehunters",2,"Leehom Wang and Kelly Rowland");\nINSERT INTO  "farm_competition" VALUES (3,"2005","MTV Asia Aid",3,"Alicia Keys");\nINSERT INTO  "farm_competition" VALUES (4,"2004","Valentine\'s Day",4,"Vanness Wu and Michelle Branch");\nINSERT INTO  "farm_competition" VALUES (5,"2003","MTV Cube",5,"Shaggy and Coco Lee");\nINSERT INTO  "farm_competition" VALUES (6,"2002","Aliens",5,"Mandy Moore and Ronan Keating");',
  1.2192101,
  'farm_competition',
  'farm'),
 ('CREATE TABLE "stadium" (\n"ID" int,\n"name" text,\n"Capacity" int,\n"City" text,\n"Country" text,\n"Opening_year" int,\nPRIMARY KEY ("ID")\